#SE NAO TIVER ENV RODA ESSE

In [ ]:

%conda create --name IBD --file environment.yml


Remove existing environment?
This will remove ALL directories contained within this specified prefix directory, including any other conda environments.

 (y/[n])? 

In [ ]:
import pandas as pd
import numpy as np

XLSX_FILE = 'gn_marco_2025.xlsx'
#read XLSX file
data = pd.read_excel(XLSX_FILE)

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import re
import os

# --- Configuração ---
# Certifique-se de que o arquivo XLSX está no mesmo diretório que o script
# ou forneça o caminho completo.
XLSX_FILE = 'gn_marco_2025.xlsx'
DB_FILE = 'gas_data.db'


# --- 1. EXTRAÇÃO (Extract) ---
print(f"Iniciando a extração do arquivo '{XLSX_FILE}'...")
try:
    df_raw = pd.read_excel(XLSX_FILE)
except FileNotFoundError:
    print(f"ERRO: O arquivo '{XLSX_FILE}' não foi encontrado.")
    print("Por favor, certifique-se de que o arquivo está no diretório correto.")
  

print("Extração concluída.")
# --- 2. TRANSFORMAÇÃO (Transform) ---
print("Iniciando a transformação dos dados...")
# Mapeamento de colunas para nomes mais curtos e compatíveis com SQL
column_mapping = {
    'Código da Instalação de Transporte': 'codigo_instalacao_transporte',
    'Nome da Instalação de Transporte': 'nome_instalacao_transporte',
    'Nome da Instalação de Gasoduto': 'nome_instalacao_gasoduto',
    'Código da Instalação de Gasoduto': 'codigo_instalacao_gasoduto',
    'Tipo da instalação de Gasoduto': 'tipo_instalacao',
    'Nome do Município da Instalação de Gasoduto': 'municipio',
    'Nome da UF da Instalação de Gasoduto': 'uf',
    'Nome do Operador da instalação de Gasoduto': 'nome_operador',
    'Código do Operador da Instalação de Gasoduto': 'codigo_operador',
    'Nome do Carregador que usa a Instalação de Gasoduto': 'nome_carregador',
    'Código do Carregador que usa a Instalação de Gasoduto': 'codigo_carregador',
    'Nome do Contrato da Instalação de Gasoduto': 'nome_contrato',
    'Nome da Variável': 'variavel_completa'
}
df = df_raw.rename(columns=column_mapping)
# Identificar colunas de data para a operação de "unpivot"
date_columns = [col for col in df.columns if isinstance(col, str) and re.match(r'^\d{4}-\d{2}-\d{2}$', col)]
id_vars = list(column_mapping.values())
# Unpivot (transformar de formato largo para longo)
df_long = pd.melt(df, id_vars=id_vars, value_vars=date_columns, var_name='data_medicao', value_name='valor')
# Limpeza de dados
df_long.replace('#N/D', np.nan, inplace=True)
df_long['valor'] = pd.to_numeric(df_long['valor'], errors='coerce')
df_long.dropna(subset=['valor'], inplace=True) # Remove linhas sem valor de medição
df_long['data_medicao'] = pd.to_datetime(df_long['data_medicao'])
# Extrair Nome da Variável e Unidade de Medida
# Regex para capturar o nome (grupo 1) e a unidade entre parênteses (grupo 2)
var_regex = re.compile(r'^(.*?)\s*\((.*)\)$')
extracted_vars = df_long['variavel_completa'].str.extract(var_regex)

df_long['nome_variavel'] = extracted_vars[0].str.strip()
df_long['unidade_medida'] = extracted_vars[1].str.strip()
# Lidar com variáveis que não têm unidade (o regex resultará em NaN)
df_long['nome_variavel'].fillna(df_long['variavel_completa'], inplace=True)
df_long['unidade_medida'].fillna('N/A', inplace=True)
# Converter tipos de dados para otimizar memória e garantir consistência
for col in ['codigo_instalacao_transporte', 'codigo_operador', 'codigo_carregador']:
    df_long[col] = pd.to_numeric(df_long[col], errors='coerce').astype('Int64')

# O código da instalação de gasoduto pode ser nulo
df_long['codigo_instalacao_gasoduto'] = pd.to_numeric(df_long['codigo_instalacao_gasoduto'], errors='coerce').astype('Int64')
print("Transformação de dados concluída.")
# --- 3. CARGA (Load) ---
print("Iniciando a carga dos dados no banco de dados SQLite...")
# Remover arquivo de banco de dados antigo, se existir, para uma execução limpa
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;") # Habilitar restrições de chave estrangeira
# DDL - Data Definition Language (Comandos para criar as tabelas)
ddl_scripts = """
CREATE TABLE Operador (
    codigo_operador INTEGER PRIMARY KEY,
    nome_operador TEXT NOT NULL UNIQUE
);
CREATE TABLE Carregador (
    codigo_carregador INTEGER PRIMARY KEY,
    nome_carregador TEXT NOT NULL UNIQUE
);
CREATE TABLE Variavel (
    id_variavel INTEGER PRIMARY KEY AUTOINCREMENT,
    nome_variavel TEXT NOT NULL,
    unidade_medida TEXT NOT NULL,
    CONSTRAINT uq_variavel_nome_unidade UNIQUE (nome_variavel, unidade_medida)
);
CREATE TABLE Instalacao_Transporte (
    codigo_instalacao_transporte INTEGER PRIMARY KEY,
    nome_instalacao_transporte TEXT NOT NULL UNIQUE,
    codigo_operador INTEGER NOT NULL,
    FOREIGN KEY (codigo_operador) REFERENCES Operador(codigo_operador)
);
CREATE TABLE Instalacao_Gasoduto (
    codigo_instalacao_gasoduto INTEGER PRIMARY KEY,
    nome_instalacao_gasoduto TEXT NOT NULL,
    tipo_instalacao TEXT NOT NULL,
    municipio TEXT NOT NULL,
    uf TEXT NOT NULL,
    codigo_instalacao_transporte INTEGER NOT NULL,
    FOREIGN KEY (codigo_instalacao_transporte) REFERENCES Instalacao_Transporte(codigo_instalacao_transporte)
);
CREATE TABLE Contrato (
    id_contrato INTEGER PRIMARY KEY AUTOINCREMENT,
    nome_contrato TEXT NOT NULL,
    codigo_instalacao_transporte INTEGER NOT NULL,
    codigo_carregador INTEGER NOT NULL,
    FOREIGN KEY (codigo_instalacao_transporte) REFERENCES Instalacao_Transporte(codigo_instalacao_transporte),
    FOREIGN KEY (codigo_carregador) REFERENCES Carregador(codigo_carregador),
    CONSTRAINT uq_contrato_transporte_carregador UNIQUE (codigo_instalacao_transporte, codigo_carregador)
);
CREATE TABLE Medicao (
    id_medicao INTEGER PRIMARY KEY AUTOINCREMENT,
    data_medicao DATE NOT NULL,
    valor REAL NOT NULL,
    id_variavel INTEGER NOT NULL,
    codigo_carregador INTEGER NOT NULL,
    codigo_instalacao_transporte INTEGER NOT NULL,
    codigo_instalacao_gasoduto INTEGER NULL,
    FOREIGN KEY (id_variavel) REFERENCES Variavel(id_variavel),
    FOREIGN KEY (codigo_carregador) REFERENCES Carregador(codigo_carregador),
    FOREIGN KEY (codigo_instalacao_transporte) REFERENCES Instalacao_Transporte(codigo_instalacao_transporte),
    FOREIGN KEY (codigo_instalacao_gasoduto) REFERENCES Instalacao_Gasoduto(codigo_instalacao_gasoduto),
    CONSTRAINT uq_medicao_unica UNIQUE (data_medicao, id_variavel, codigo_carregador, codigo_instalacao_transporte, codigo_instalacao_gasoduto)
);
"""
cursor.executescript(ddl_scripts)
print("Esquema do banco de dados criado.")
# Povoar tabelas de dimensão (a ordem é importante devido às chaves estrangeiras)

# Operador
df_operador = df_long[['codigo_operador', 'nome_operador']].drop_duplicates().dropna(subset=['codigo_operador'])
df_operador.to_sql('Operador', conn, if_exists='append', index=False)
# Carregador
df_carregador = df_long[['codigo_carregador', 'nome_carregador']].drop_duplicates().dropna(subset=['codigo_carregador'])
df_carregador.to_sql('Carregador', conn, if_exists='append', index=False)
# Variavel (com chave substituta)
df_variavel = df_long[['nome_variavel', 'unidade_medida']].drop_duplicates().dropna()
df_variavel.to_sql('Variavel', conn, if_exists='append', index=False)
df_variavel_com_id = pd.read_sql("SELECT id_variavel, nome_variavel, unidade_medida FROM Variavel", conn)
# Instalacao_Transporte
df_instalacao_transporte = df_long[['codigo_instalacao_transporte', 'nome_instalacao_transporte', 'codigo_operador']].drop_duplicates().dropna(subset=['codigo_instalacao_transporte'])
df_instalacao_transporte.to_sql('Instalacao_Transporte', conn, if_exists='append', index=False)
# Instalacao_Gasoduto
df_instalacao_gasoduto = df_long[['codigo_instalacao_gasoduto', 'nome_instalacao_gasoduto', 'tipo_instalacao', 'municipio', 'uf', 'codigo_instalacao_transporte']].drop_duplicates().dropna(subset=['codigo_instalacao_gasoduto'])
df_instalacao_gasoduto.to_sql('Instalacao_Gasoduto', conn, if_exists='append', index=False)
# Contrato
df_contrato = df_long[['nome_contrato', 'codigo_instalacao_transporte', 'codigo_carregador']].drop_duplicates().dropna()
df_contrato.to_sql('Contrato', conn, if_exists='append', index=False)

print("Tabelas de dimensão povoadas.")
# Preparar e povoar a tabela de fatos (Medicao)
# Juntar com a tabela de variáveis para obter a chave estrangeira 'id_variavel'
df_medicao_final = pd.merge(df_long, df_variavel_com_id, on=['nome_variavel', 'unidade_medida'], how='left')
# Selecionar e preparar as colunas finais para a tabela Medicao
colunas_medicao = [
    'data_medicao',
    'valor',
    'id_variavel',
    'codigo_carregador',
    'codigo_instalacao_transporte',
    'codigo_instalacao_gasoduto'
]
df_medicao_final = df_medicao_final[colunas_medicao]

# Garantir que os tipos de dados estão corretos antes de inserir
df_medicao_final['id_variavel'] = df_medicao_final['id_variavel'].astype(int)
df_medicao_final['codigo_carregador'] = df_medicao_final['codigo_carregador'].astype(int)
df_medicao_final['codigo_instalacao_transporte'] = df_medicao_final['codigo_instalacao_transporte'].astype(int)

# A coluna 'codigo_instalacao_gasoduto' pode conter nulos
df_medicao_final['codigo_instalacao_gasoduto'] = df_medicao_final['codigo_instalacao_gasoduto'].astype('Int64')
# Remover duplicatas que possam ter surgido (embora a restrição UNIQUE na tabela já impeça isso)
df_medicao_final.drop_duplicates(inplace=True)
df_medicao_final.to_sql('Medicao', conn, if_exists='append', index=False)
print("Tabela de fatos (Medicao) povoada.")
# Finalizar a transação e fechar a conexão
conn.commit()
conn.close()

print(f"\nProcesso concluído com sucesso! O banco de dados '{DB_FILE}' foi criado e populado.")


In [3]:
import sqlite3
import pandas as pd

# Nome do seu arquivo de banco de dados
db_file = 'gas_data.db'

# Conectar ao banco de dados
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# Query para listar todas as tabelas no banco de dados
query_tables = "SELECT name FROM sqlite_master WHERE type='table';"
cursor.execute(query_tables)

# Obter os nomes das tabelas
table_names = [table[0] for table in cursor.fetchall()]

print("Tabelas encontradas no banco de dados:")
print(table_names)

# Fechar a conexão
# conn.close() # Manteremos aberta para os próximos passos

Tabelas encontradas no banco de dados:
['Operador', 'Carregador', 'Variavel', 'sqlite_sequence', 'Instalacao_Transporte', 'Instalacao_Gasoduto', 'Contrato', 'Medicao']


In [4]:
print("\n--- Estrutura das Tabelas ---")
for table_name in table_names:
    print(f"\n[ Tabela: {table_name} ]")
    
    # Usamos PRAGMA table_info para obter o schema
    query_schema = f"PRAGMA table_info('{table_name}');"
    
    # Usamos o pandas para ler o resultado da query e formatar bem
    schema_df = pd.read_sql_query(query_schema, conn)
    print(schema_df)


--- Estrutura das Tabelas ---

[ Tabela: Operador ]
   cid             name     type  notnull dflt_value  pk
0    0  codigo_operador  INTEGER        0       None   1
1    1    nome_operador     TEXT        1       None   0

[ Tabela: Carregador ]
   cid               name     type  notnull dflt_value  pk
0    0  codigo_carregador  INTEGER        0       None   1
1    1    nome_carregador     TEXT        1       None   0

[ Tabela: Variavel ]
   cid            name     type  notnull dflt_value  pk
0    0     id_variavel  INTEGER        0       None   1
1    1   nome_variavel     TEXT        1       None   0
2    2  unidade_medida     TEXT        1       None   0

[ Tabela: sqlite_sequence ]
   cid  name type  notnull dflt_value  pk
0    0  name             0       None   0
1    1   seq             0       None   0

[ Tabela: Instalacao_Transporte ]
   cid                          name     type  notnull dflt_value  pk
0    0  codigo_instalacao_transporte  INTEGER        0       None   1

In [5]:
#salva a env atual em um arquivo usando %
%conda env export > environment.yml



Note: you may need to restart the kernel to use updated packages.
